# Laboratorio de Modelado y Simulacion
## Metodos Numericos - Parcial 1 (Clases 1-7)

**Requisitos:** `pip install numpy matplotlib ipywidgets sympy`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from sympy import sympify, lambdify, Symbol, pi as sym_pi

plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100, 'font.size': 10})
x_sym, y_sym = Symbol('x'), Symbol('y')

def pfx(e): return lambdify(x_sym, sympify(e), modules=['numpy'])
def pfxy(e): return lambdify((x_sym, y_sym), sympify(e), modules=['numpy'])
def fmt(v, d=8):
    if isinstance(v, (int,float,np.floating)):
        return 'NaN' if np.isnan(v) else f'{v:.{d}f}'
    return str(v)

def make_ui(fields, btn_text, run_fn):
    ws = {}
    for name, default, width in fields:
        if isinstance(default, float): ws[name] = widgets.FloatText(value=default, description=name, layout=widgets.Layout(width=f'{width}px'))
        elif isinstance(default, int): ws[name] = widgets.IntText(value=default, description=name, layout=widgets.Layout(width=f'{width}px'))
        else: ws[name] = widgets.Text(value=str(default), description=name, layout=widgets.Layout(width=f'{width}px'))
    btn = widgets.Button(description=btn_text, button_style='primary')
    out = widgets.Output()
    def _run(_): 
        with out: clear_output(wait=True); run_fn(ws)
    btn.on_click(_run)
    items = list(ws.values())
    display(widgets.VBox(items + [btn, out]))
    return ws, out

print('OK - Librerias cargadas')

---
## Clase 1: Biseccion

In [ ]:
def run_biseccion(w):
    f=pfx(w['f(x)'].value); a,b=w['a'].value,w['b'].value; tol,mx=w['Tol'].value,w['Max'].value
    if f(a)*f(b)>0: print('Error: f(a)*f(b)>0'); return
    oA,oB,data=a,b,[]
    for i in range(1,mx+1):
        c=(a+b)/2; fc=f(c); err=abs(b-a)/2; data.append([i,a,b,c,f(a),fc,err])
        if abs(fc)<1e-15 or err<tol: break
        if f(a)*fc<0: b=c
        else: a=c
    L=data[-1]; print(f'Raiz: x={fmt(L[3],10)} | f(x)={fmt(L[5])} | Iter:{L[0]} | Err:{fmt(L[6])}')
    print(f'{"n":>3} {"a":>13} {"b":>13} {"c":>13} {"f(c)":>13} {"Error":>13}'); print('-'*75)
    for r in data: print(f'{r[0]:3d} {r[1]:13.8f} {r[2]:13.8f} {r[3]:13.8f} {r[5]:13.8f} {r[6]:13.8f}')
    fig,ax=plt.subplots(); xs=np.linspace(oA-0.5,oB+0.5,300); ax.plot(xs,f(xs),'b-',lw=2); ax.axhline(0,color='gray',lw=0.5)
    ax.plot([r[3] for r in data],[r[5] for r in data],'ro',ms=4); ax.axvline(L[3],color='green',ls='--',alpha=0.7)
    ax.set_title('Biseccion'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','x**3-x-2',350),('a',1.0,150),('b',2.0,150),('Tol',0.0001,180),('Max',50,150)],'Ejecutar',run_biseccion)

---
## Clase 1: Punto Fijo

In [ ]:
def run_pf(w):
    g=pfx(w['g(x)'].value); x=w['x0'].value; tol,mx=w['Tol'].value,w['Max'].value; data=[]
    for i in range(1,mx+1):
        xn=float(g(x)); err=abs(xn-x); data.append([i,x,xn,err])
        if not np.isfinite(xn): print('Diverge'); return
        x=xn
        if err<tol: break
    print(f'Punto fijo: {fmt(x,10)} | Iter:{data[-1][0]}')
    print(f'{"n":>3} {"x_n":>13} {"g(x_n)":>13} {"Error":>13}'); print('-'*48)
    for r in data: print(f'{r[0]:3d} {r[1]:13.8f} {r[2]:13.8f} {r[3]:13.8f}')
    fig,ax=plt.subplots(); aX=[r[1] for r in data]+[r[2] for r in data]; mn,mx2=min(aX)-0.5,max(aX)+0.5
    xs=np.linspace(mn,mx2,300); ax.plot(xs,g(xs),'b-',lw=2,label='g(x)'); ax.plot(xs,xs,'--',color='gray',label='y=x')
    ax.plot([r[1] for r in data],[r[2] for r in data],'ro',ms=4); ax.legend(); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('g(x)','(x+2)**(1/3)',350),('x0',1.5,150),('Tol',0.0001,180),('Max',50,150)],'Ejecutar',run_pf)

---
## Clase 2: Newton-Raphson

In [ ]:
def run_nr(w):
    f=pfx(w['f(x)'].value); df=pfx(w["f'(x)"].value); x=w['x0'].value; tol,mx=w['Tol'].value,w['Max'].value; data=[]
    for i in range(1,mx+1):
        fv,dv=f(x),df(x)
        if abs(dv)<1e-15: print('f\'~0'); return
        xn=x-fv/dv; err=abs(xn-x); data.append([i,x,fv,dv,xn,err]); x=xn
        if err<tol: break
    print(f'Raiz: {fmt(x,10)} | f(x)={fmt(f(x))} | Iter:{data[-1][0]}')
    print(f'{"n":>3} {"x_n":>13} {"f(x_n)":>13} {"f\'(x_n)":>13} {"x_n+1":>13} {"Error":>13}'); print('-'*78)
    for r in data: print(f'{r[0]:3d} {r[1]:13.8f} {r[2]:13.8f} {r[3]:13.8f} {r[4]:13.8f} {r[5]:13.8f}')
    fig,ax=plt.subplots(); rng=max(abs(max(r[1] for r in data)-min(r[1] for r in data))*2,2)
    xs=np.linspace(x-rng,x+rng,300); ax.plot(xs,f(xs),'b-',lw=2); ax.axhline(0,color='gray',lw=0.5)
    ax.plot([r[1] for r in data],[r[2] for r in data],'ro',ms=4); ax.set_title('Newton-Raphson'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','x**3-x-2',350),("f'(x)",'3*x**2-1',350),('x0',1.5,150),('Tol',0.0001,180),('Max',50,150)],'Ejecutar',run_nr)

---
## Clase 2: Aceleracion de Aitken

In [ ]:
def run_aitken(w):
    g=pfx(w['g(x)'].value); x=w['x0'].value; tol,mx=w['Tol'].value,w['Max'].value
    seq=[x]
    for _ in range(mx+2): x=float(g(x)); seq.append(x) if np.isfinite(x) else None
    aitken=[]
    for n in range(len(seq)-2):
        s0,s1,s2=seq[n],seq[n+1],seq[n+2]; d=s2-2*s1+s0
        if abs(d)<1e-15: break
        aitken.append(s0-(s1-s0)**2/d)
    print(f'PF final: {fmt(seq[-1])} | Aitken: {fmt(aitken[-1] if aitken else 0, 10)}')
    print(f'{"n":>3} {"x_n PF":>14} {"x_n* Aitken":>14} {"Err PF":>14} {"Err Aitken":>14}'); print('-'*65)
    pfE,aE=[],[]
    for n in range(min(len(seq),len(aitken)+2,mx)):
        ep=abs(seq[n]-seq[n-1]) if n>0 else None; ea=abs(aitken[n]-aitken[n-1]) if n>0 and n<len(aitken) else None
        if ep: pfE.append(ep)
        if ea: aE.append(ea)
        print(f'{n:3d} {seq[n]:14.8f} {aitken[n] if n<len(aitken) else "":>14} {ep if ep else "":>14} {ea if ea else "":>14}')
        if ea and ea<tol: break
    fig,ax=plt.subplots()
    if pfE: ax.semilogy(range(1,len(pfE)+1),pfE,'r-o',ms=3,label='Err PF')
    if aE: ax.semilogy(range(1,len(aE)+1),aE,'g-o',ms=3,label='Err Aitken')
    ax.legend(); ax.set_title('Convergencia PF vs Aitken'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('g(x)','(x+2)**(1/3)',350),('x0',1.5,150),('Tol',0.0001,180),('Max',50,150)],'Ejecutar',run_aitken)

---
## Clase 3: Interpolacion de Lagrange

In [ ]:
def run_lag(w):
    xv=[float(v) for v in w['x'].value.split(',')]; yv=[float(v) for v in w['y'].value.split(',')]; ex=w['Eval'].value; n=len(xv)
    def lag(xp):
        s=0
        for i in range(n):
            l=1
            for j in range(n):
                if i!=j: l*=(xp-xv[j])/(xv[i]-xv[j])
            s+=yv[i]*l
        return s
    print(f'P({ex}) = {fmt(lag(ex),10)}')
    fig,ax=plt.subplots(); xs=np.linspace(min(xv)-0.5,max(xv)+0.5,300)
    ax.plot(xs,[lag(xi) for xi in xs],'b-',lw=2,label='P(x)'); ax.plot(xv,yv,'ro',ms=6,label='Puntos'); ax.plot(ex,lag(ex),'gs',ms=8)
    ax.legend(); ax.grid(True,alpha=0.3); ax.set_title('Lagrange'); plt.tight_layout(); plt.show()

make_ui([('x','0,1,2,3',350),('y','1,2.718,7.389,20.086',350),('Eval',1.5,180)],'Interpolar',run_lag)

---
## Clase 3: Newton Diferencias Divididas

In [ ]:
def run_ni(w):
    xv=[float(v) for v in w['x'].value.split(',')]; yv=[float(v) for v in w['y'].value.split(',')]; ex=w['Eval'].value; n=len(xv)
    dd=[[0.0]*n for _ in range(n)]
    for i in range(n): dd[i][0]=yv[i]
    for j in range(1,n):
        for i in range(n-j): dd[i][j]=(dd[i+1][j-1]-dd[i][j-1])/(xv[i+j]-xv[i])
    def np_(xp):
        r,p=dd[0][0],1.0
        for j in range(1,n): p*=(xp-xv[j-1]); r+=dd[0][j]*p
        return r
    print(f'P({ex}) = {fmt(np_(ex),10)}')
    print('\nDiferencias Divididas:')
    for i in range(n): print(f'  x={fmt(xv[i],3)}', ' '.join(fmt(dd[i][j],6) for j in range(n-i)))
    fig,ax=plt.subplots(); xs=np.linspace(min(xv)-1,max(xv)+1,300)
    ax.plot(xs,[np_(xi) for xi in xs],'b-',lw=2); ax.plot(xv,yv,'ro',ms=6); ax.plot(ex,np_(ex),'gs',ms=8)
    ax.set_title('Newton Dif. Div.'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('x','1,2,4,7',350),('y','1,4,16,49',350),('Eval',3.0,180)],'Interpolar',run_ni)

---
## Clase 3: Derivacion Numerica

In [ ]:
def run_der(w):
    f=pfx(w['f(x)'].value); x0=w['x0'].value; h0=w['h'].value
    exact=pfx(w["f'exact"].value) if w["f'exact"].value else None; ev=float(exact(x0)) if exact else None
    fwd=(f(x0+h0)-f(x0))/h0; bwd=(f(x0)-f(x0-h0))/h0; ctr=(f(x0+h0)-f(x0-h0))/(2*h0)
    d2=(f(x0+h0)-2*f(x0)+f(x0-h0))/(h0**2)
    print(f'Forward: {fmt(fwd,10)} | Backward: {fmt(bwd,10)} | Central: {fmt(ctr,10)} | f\'\': {fmt(d2,10)}')
    if ev: print(f'Exacto: {fmt(ev,10)} | Error central: {fmt(abs(ctr-ev))}')
    hs,eF,eC=[],[],[]
    print(f'\n{"h":>12} {"Forward":>14} {"Central":>14} {"f\'\':":<14}',end=''); 
    if ev: print(f' {"Err F":>12} {"Err C":>12}'); 
    else: print()
    for k in range(12):
        h=1/2**k; hs.append(h)
        fw=(f(x0+h)-f(x0))/h; ct=(f(x0+h)-f(x0-h))/(2*h); dd=(f(x0+h)-2*f(x0)+f(x0-h))/(h**2)
        ef=abs(fw-ev) if ev else 0; ec=abs(ct-ev) if ev else 0; eF.append(ef); eC.append(ec)
        print(f'{h:12.6f} {fw:14.8f} {ct:14.8f} {dd:14.8f}',end='')
        if ev: print(f' {ef:12.2e} {ec:12.2e}')
        else: print()
    if ev:
        fig,ax=plt.subplots(); ax.loglog(hs,eF,'r-o',ms=3,label='Err Fwd O(h)'); ax.loglog(hs,eC,'g-o',ms=3,label='Err Ctr O(h^2)')
        ax.legend(); ax.set_xlabel('h'); ax.set_title('Error vs h'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','sin(x)',350),('x0',1.0,150),('h',0.1,150),("f'exact",'cos(x)',300)],'Calcular',run_der)

---
## Clase 5: Regla de Rectangulos

In [ ]:
def run_rect(w):
    f=pfx(w['f(x)'].value); a,b=w['a'].value,w['b'].value; n=w['n'].value; tipo=w['Tipo'].value.upper()
    h=(b-a)/n; xs,ys=[],[]
    for i in range(n):
        xp = a+i*h if tipo=='L' else (a+(i+1)*h if tipo=='R' else a+(i+0.5)*h)
        xs.append(xp); ys.append(float(f(xp)))
    integral=h*sum(ys); nm={'L':'Izquierda','R':'Derecha','M':'Punto Medio'}
    print(f'Integral ({nm.get(tipo,tipo)}) = {fmt(integral,10)} | h={fmt(h)} | n={n}')
    fig,ax=plt.subplots(); xf=np.linspace(a,b,300); ax.plot(xf,f(xf),'b-',lw=2)
    for i in range(n):
        x0,x1=a+i*h,a+(i+1)*h; ax.fill([x0,x1,x1,x0],[0,0,ys[i],ys[i]],alpha=0.15,color='blue')
    ax.plot(xs,ys,'ro',ms=3); ax.set_title(f'Rectangulos ({nm.get(tipo,tipo)})'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','sin(x)',350),('a',0.0,150),('b',3.14159,150),('n',10,120),('Tipo','M',120)],'Calcular',run_rect)

---
## Clase 5: Trapecio

In [ ]:
def run_trap(w):
    f=pfx(w['f(x)'].value); a,b,n=w['a'].value,w['b'].value,w['n'].value; h=(b-a)/n
    xs=[a+i*h for i in range(n+1)]; ys=[float(f(x)) for x in xs]
    s=ys[0]+ys[-1]+sum(2*ys[i] for i in range(1,n)); integral=(h/2)*s
    print(f'Integral = {fmt(integral,10)} | h={fmt(h)} | n={n}')
    fig,ax=plt.subplots(); xf=np.linspace(a,b,300); ax.plot(xf,f(xf),'b-',lw=2)
    for i in range(n): ax.fill([xs[i],xs[i],xs[i+1],xs[i+1]],[0,ys[i],ys[i+1],0],alpha=0.12,color='blue')
    ax.set_title('Trapecio'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','sin(x)',350),('a',0.0,150),('b',3.14159,150),('n',10,120)],'Calcular',run_trap)

---
## Clase 5: Simpson 1/3

In [ ]:
def run_simp(w):
    f=pfx(w['f(x)'].value); a,b=w['a'].value,w['b'].value; n=w['n(par)'].value
    if n%2: n+=1
    h=(b-a)/n; xs=[a+i*h for i in range(n+1)]; ys=[float(f(x)) for x in xs]
    s=ys[0]+ys[-1]+sum((4 if i%2 else 2)*ys[i] for i in range(1,n)); integral=(h/3)*s
    print(f'Integral = {fmt(integral,10)} | h={fmt(h)} | n={n}')
    fig,ax=plt.subplots(); xf=np.linspace(a,b,300); ax.plot(xf,f(xf),'b-',lw=2); ax.fill_between(xf,0,f(xf),alpha=0.12)
    ax.plot(xs,ys,'ro',ms=4); ax.set_title('Simpson 1/3'); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

make_ui([('f(x)','exp(-x**2)',350),('a',0.0,150),('b',1.0,150),('n(par)',10,120)],'Calcular',run_simp)

---
## Clase 6: Montecarlo 1D + 2D

In [ ]:
def run_mc(w):
    f=pfx(w['f(x)'].value); a,b,N=w['a'].value,w['b'].value,w['N'].value
    xs=np.random.uniform(a,b,N); vals=np.array([float(f(x)) for x in xs])
    mean=np.mean(vals); var=np.var(vals,ddof=1); se=np.sqrt(var/N)
    integral=(b-a)*mean; err=(b-a)*se; ci=1.96*err
    print(f'MC 1D = {fmt(integral,10)}')
    print(f'Varianza: {fmt(var)} | Err estandar: {fmt(err)}')
    print(f'IC 95%: [{fmt(integral-ci)}, {fmt(integral+ci)}]')
    cum=np.cumsum(vals); ns=np.arange(1,N+1); conv=(b-a)*cum/ns
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
    step=max(1,N//200); ax1.plot(ns[::step],conv[::step],'b-',lw=1); ax1.set_title('Convergencia MC'); ax1.set_xlabel('N'); ax1.grid(True,alpha=0.3)
    # Histogram
    ax2.hist(vals,bins=50,alpha=0.7,color='steelblue',edgecolor='white'); ax2.set_title('Distribucion f(x)'); ax2.grid(True,alpha=0.3)
    plt.tight_layout(); plt.show()

make_ui([('f(x)','sin(x)',350),('a',0.0,150),('b',3.14159,150),('N',10000,180)],'Estimar',run_mc)

In [ ]:
def run_mc2d(w):
    f2=pfxy(w['f(x,y)'].value); ax,bx,ay,by=w['ax'].value,w['bx'].value,w['ay'].value,w['by'].value; N=w['N'].value
    vol=(bx-ax)*(by-ay); xr=np.random.uniform(ax,bx,N); yr=np.random.uniform(ay,by,N)
    vals=np.array([float(f2(xr[i],yr[i])) for i in range(N)])
    m=np.mean(vals); var=np.var(vals,ddof=1); i2=vol*m; se=vol*np.sqrt(var/N)
    print(f'MC 2D = {fmt(i2,10)} | IC95%: [{fmt(i2-1.96*se)}, {fmt(i2+1.96*se)}] | N={N}')

make_ui([('f(x,y)','x**2+y**2',350),('ax',0.0,120),('bx',1.0,120),('ay',0.0,120),('by',1.0,120),('N',50000,150)],'Estimar 2D',run_mc2d)

---
## Clase 7: Euler

In [ ]:
def run_euler(w):
    f=pfxy(w['f(x,y)'].value); x,y=w['x0'].value,w['y0'].value; xf,h=w['xf'].value,w['h'].value
    data=[[0,x,y]]; i=1
    while x+h/2<xf: y+=h*float(f(x,y)); x=round(x+h,10); data.append([i,x,y]); i+=1
    print(f'y({fmt(xf,4)}) = {fmt(data[-1][2],10)} | Pasos:{len(data)-1}')
    print(f'{"i":>3} {"x":>10} {"y":>14}'); print('-'*30)
    for r in data: print(f'{r[0]:3d} {r[1]:10.6f} {r[2]:14.8f}')
    fig,ax=plt.subplots(); ax.plot([r[1] for r in data],[r[2] for r in data],'b-o',ms=4,label='Euler')
    ax.legend(); ax.grid(True,alpha=0.3); ax.set_title('Euler'); plt.tight_layout(); plt.show()

make_ui([('f(x,y)','x+y',350),('x0',0.0,120),('y0',1.0,120),('xf',2.0,120),('h',0.2,120)],'Calcular',run_euler)

---
## Clase 7: Euler Modificado (Heun)

In [ ]:
def run_heun(w):
    f=pfxy(w['f(x,y)'].value); x,y=w['x0'].value,w['y0'].value; xf,h=w['xf'].value,w['h'].value
    data=[[0,x,y,'-','-']]; i=1
    while x+h/2<xf:
        f0=float(f(x,y)); yP=y+h*f0; x1=round(x+h,10); f1=float(f(x1,yP))
        y=y+h/2*(f0+f1); x=x1; data.append([i,x,y,fmt(yP),fmt(f0)]); i+=1
    print(f'y({fmt(xf,4)}) = {fmt(data[-1][2],10)} | Pasos:{len(data)-1}')
    print(f'{"i":>3} {"x":>10} {"y":>14} {"y*(pred)":>14} {"f0":>12}'); print('-'*58)
    for r in data: print(f'{r[0]:3d} {r[1]:10.6f} {r[2]:14.8f} {r[3]:>14} {r[4]:>12}')
    fig,ax=plt.subplots(); ax.plot([r[1] for r in data],[r[2] for r in data],'b-o',ms=4,label='Heun')
    ax.legend(); ax.grid(True,alpha=0.3); ax.set_title('Euler Modificado (Heun)'); plt.tight_layout(); plt.show()

make_ui([('f(x,y)','x+y',350),('x0',0.0,120),('y0',1.0,120),('xf',2.0,120),('h',0.2,120)],'Calcular',run_heun)

---
## Clase 7: Taylor Orden 2

In [ ]:
def run_taylor(w):
    f=pfxy(w['f(x,y)'].value); df=pfxy(w["f'(x,y)"].value); x,y=w['x0'].value,w['y0'].value; xf,h=w['xf'].value,w['h'].value
    data=[[0,x,y]]; i=1
    while x+h/2<xf:
        fv,dfv=float(f(x,y)),float(df(x,y)); y+=h*fv+(h**2/2)*dfv; x=round(x+h,10)
        data.append([i,x,y]); i+=1
    print(f'y({fmt(xf,4)}) = {fmt(data[-1][2],10)} | Pasos:{len(data)-1}')
    print(f'{"i":>3} {"x":>10} {"y":>14}'); print('-'*30)
    for r in data: print(f'{r[0]:3d} {r[1]:10.6f} {r[2]:14.8f}')
    fig,ax=plt.subplots(); ax.plot([r[1] for r in data],[r[2] for r in data],'b-o',ms=4,label='Taylor O(2)')
    ax.legend(); ax.grid(True,alpha=0.3); ax.set_title('Taylor Orden 2'); plt.tight_layout(); plt.show()

make_ui([('f(x,y)','x+y',350),("f'(x,y)",'1+x+y',350),('x0',0.0,120),('y0',1.0,120),('xf',2.0,120),('h',0.2,120)],'Calcular',run_taylor)

---
## Clase 7: Runge-Kutta (RK4)

In [ ]:
def run_rk4(w):
    f=pfxy(w['f(x,y)'].value); x,y=w['x0'].value,w['y0'].value; xf,h=w['xf'].value,w['h'].value
    data=[[0,x,y,'-','-','-','-']]; i=1
    while x+h/2<xf:
        k1=h*float(f(x,y)); k2=h*float(f(x+h/2,y+k1/2)); k3=h*float(f(x+h/2,y+k2/2)); k4=h*float(f(x+h,y+k3))
        y+=(k1+2*k2+2*k3+k4)/6; x=round(x+h,10); data.append([i,x,y,k1,k2,k3,k4]); i+=1
    print(f'y({fmt(xf,4)}) = {fmt(data[-1][2],10)} | Pasos:{len(data)-1}')
    print(f'{"i":>3} {"x":>10} {"y":>14} {"k1":>12} {"k2":>12} {"k3":>12} {"k4":>12}'); print('-'*82)
    for r in data:
        ks=' '.join(f'{v:12.8f}' if isinstance(v,float) else f'{v:>12}' for v in r[3:])
        print(f'{r[0]:3d} {r[1]:10.6f} {r[2]:14.8f} {ks}')
    fig,ax=plt.subplots(); ax.plot([r[1] for r in data],[r[2] for r in data],'b-o',ms=4,label='RK4')
    ax.legend(); ax.grid(True,alpha=0.3); ax.set_title('Runge-Kutta (RK4)'); plt.tight_layout(); plt.show()

make_ui([('f(x,y)','x+y',350),('x0',0.0,120),('y0',1.0,120),('xf',2.0,120),('h',0.2,120)],'Calcular',run_rk4)

---
## Clase 7: Comparacion Euler vs Heun vs Taylor vs RK4

In [ ]:
def run_cmp(w):
    fe=pfxy(w['f(x,y)'].value); dfe=pfxy(w["f'"].value); x0,y0=w['x0'].value,w['y0'].value; xf,h=w['xf'].value,w['h'].value
    methods={'Euler':[],'Heun':[],'Taylor':[],'RK4':[]}
    # Euler
    x,y=x0,y0; methods['Euler'].append((x,y))
    while x+h/2<xf: y+=h*float(fe(x,y)); x=round(x+h,10); methods['Euler'].append((x,y))
    # Heun
    x,y=x0,y0; methods['Heun'].append((x,y))
    while x+h/2<xf: f0=float(fe(x,y)); yP=y+h*f0; x1=round(x+h,10); y=y+h/2*(f0+float(fe(x1,yP))); x=x1; methods['Heun'].append((x,y))
    # Taylor
    x,y=x0,y0; methods['Taylor'].append((x,y))
    while x+h/2<xf: y+=h*float(fe(x,y))+(h**2/2)*float(dfe(x,y)); x=round(x+h,10); methods['Taylor'].append((x,y))
    # RK4
    x,y=x0,y0; methods['RK4'].append((x,y))
    while x+h/2<xf:
        k1=h*float(fe(x,y)); k2=h*float(fe(x+h/2,y+k1/2)); k3=h*float(fe(x+h/2,y+k2/2)); k4=h*float(fe(x+h,y+k3))
        y+=(k1+2*k2+2*k3+k4)/6; x=round(x+h,10); methods['RK4'].append((x,y))
    for name,pts in methods.items(): print(f'{name:8s}: y({fmt(xf,2)}) = {fmt(pts[-1][1],10)}')
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
    colors={'Euler':'red','Heun':'orange','Taylor':'purple','RK4':'blue'}
    rk4_y = {p[0]:p[1] for p in methods['RK4']}
    for name,pts in methods.items():
        ax1.plot([p[0] for p in pts],[p[1] for p in pts],'-o',ms=3,color=colors[name],label=name)
        if name!='RK4':
            errs=[abs(p[1]-rk4_y.get(p[0],p[1])) for p in pts if p[0] in rk4_y]
            ax2.plot([p[0] for p in pts if p[0] in rk4_y],errs,'-o',ms=3,color=colors[name],label=f'|{name}-RK4|')
    ax1.legend(); ax1.grid(True,alpha=0.3); ax1.set_title('Soluciones')
    ax2.legend(); ax2.grid(True,alpha=0.3); ax2.set_title('Diferencia vs RK4')
    plt.tight_layout(); plt.show()

make_ui([('f(x,y)','x+y',350),("f'",'1+x+y',350),('x0',0.0,120),('y0',1.0,120),('xf',2.0,120),('h',0.2,120)],'Comparar',run_cmp)

---
## Clase 7: EDP Diferencias Finitas (Laplace/Poisson)

In [ ]:
def run_edp(w):
    N=w['N'].value; h=1/(N+1); h2=h*h
    def evb(expr,val): return float(sympify(expr).subs({x_sym:val, Symbol('pi'):float(sym_pi)}))
    u=np.zeros((N+2,N+2))
    for i in range(N+2):
        u[i,0]=evb(w['u_inf'].value,i*h); u[i,N+1]=evb(w['u_sup'].value,i*h)
        u[0,i]=evb(w['u_izq'].value,i*h); u[N+1,i]=evb(w['u_der'].value,i*h)
    fsrc=pfxy(w['f(x,y)'].value) if w['f(x,y)'].value.strip()!='0' else None
    for _ in range(5000):
        mx=0
        for i in range(1,N+1):
            for j in range(1,N+1):
                fv=float(fsrc(i*h,j*h)) if fsrc else 0
                nv=(u[i-1,j]+u[i+1,j]+u[i,j-1]+u[i,j+1]+h2*fv)/4
                mx=max(mx,abs(nv-u[i,j])); u[i,j]=nv
        if mx<1e-6: break
    mid=N//2+1
    print(f'Resuelto {N}x{N} puntos | u(0.5,0.5) ~ {fmt(u[mid,mid])}')
    fig,ax=plt.subplots(figsize=(8,6))
    im=ax.imshow(u.T,origin='lower',extent=[0,1,0,1],cmap='RdYlBu_r',aspect='equal')
    fig.colorbar(im,ax=ax); ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('u(x,y)')
    plt.tight_layout(); plt.show()

make_ui([('f(x,y)','0',350),('N',10,120),('u_inf','sin(pi*x)',250),('u_sup','0',250),('u_izq','0',250),('u_der','0',250)],'Resolver',run_edp)